# Learn2Branch Set Cover 100k Kaggle Run

Before running: enable **Internet** and choose a **GPU accelerator** in Kaggle notebook settings.

Outputs/checkpoints are written to `/kaggle/working/learn2branch_setcover_100k`.

Important Kaggle persistence rule: `/kaggle/working` is saved only when you **Save Version / Commit** the notebook output. Inside one live session, rerunning Cell 2 resumes from local files. Across sessions, attach a previous saved output as an input dataset if you want the runner to restore old archives/checkpoints.


In [ ]:
%%bash
# Cell 1: Install Kaggle runtime environment
set -euo pipefail
cd /kaggle/working

if [ ! -x /kaggle/working/bin/micromamba ]; then
  mkdir -p /kaggle/working/bin
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /kaggle/working/bin --strip-components=1 bin/micromamba
fi

export MAMBA_ROOT_PREFIX=/kaggle/working/micromamba
PY=/kaggle/working/micromamba/envs/l2b/bin/python

if [ ! -x "$PY" ]; then
  /kaggle/working/bin/micromamba create -y -n l2b -c conda-forge python=3.11 ecole pyscipopt numpy scipy pandas matplotlib pip
else
  /kaggle/working/bin/micromamba install -y -n l2b -c conda-forge python=3.11 ecole pyscipopt numpy scipy pandas matplotlib pip
fi

"$PY" -m pip install -U pip
"$PY" - <<'CHECKPY' || "$PY" -m pip install torch torch-geometric
import torch, torch_geometric
CHECKPY

"$PY" - <<'CHECKPY'
import ecole, torch, torch_geometric, pyscipopt, sys
print('python:', sys.executable, sys.version)
print('ecole:', ecole.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('torch_geometric:', torch_geometric.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
CHECKPY


In [ ]:
# Cell 2: Run or resume the pipeline with streamed output
import os, subprocess

artifact_dir = "/kaggle/working/learn2branch_setcover_100k"
python_bin = "/kaggle/working/micromamba/envs/l2b/bin/python"
os.makedirs(artifact_dir, exist_ok=True)

print("downloading Kaggle runner", flush=True)
subprocess.run([
    "curl", "-fL",
    "https://raw.githubusercontent.com/skanda-vyas-srinivasan/Branch-Learning-Distance-Threshold/main/kaggle_learn2branch_runner.py",
    "-o", "/kaggle/working/kaggle_learn2branch_runner.py",
], check=True)

print("runner downloaded", flush=True)
print(open("/kaggle/working/kaggle_learn2branch_runner.py").read().splitlines()[0:5], flush=True)

log_path = os.path.join(artifact_dir, "runner.log")
print("starting runner", flush=True)

with open(log_path, "a", buffering=1) as log:
    p = subprocess.Popen(
        [python_bin, "-u", "/kaggle/working/kaggle_learn2branch_runner.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "L2B_ROOT": "/kaggle/working", "L2B_ARTIFACT_DIR": artifact_dir},
    )
    for line in p.stdout:
        print(line, end="", flush=True)
        log.write(line)
    rc = p.wait()

print("runner exited", rc, flush=True)
if rc:
    raise RuntimeError(f"runner failed with exit code {rc}")


In [ ]:
%%bash
# Cell 3: Inspect progress and output artifacts
set -euo pipefail
ART=/kaggle/working/learn2branch_setcover_100k
SAMPLES=/kaggle/working/learn2branch-ecole/data/samples/setcover/500r_1000c_0.05d

echo "Artifacts:"
ls -lh "$ART" 2>/dev/null || true

echo
echo "Sample counts:"
printf 'train samples: '
find "$SAMPLES/train" -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l
printf 'valid samples: '
find "$SAMPLES/valid" -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l

echo
echo "Last runner log lines:"
tail -n 80 "$ART/runner.log" 2>/dev/null || true


In [ ]:
%%bash
# Cell 4: Emergency local checkpoint in /kaggle/working
# Run only when Cell 2 is stopped. Then use Kaggle Save Version to persist /kaggle/working.
set -euo pipefail
ART=/kaggle/working/learn2branch_setcover_100k
SRC=/kaggle/working/learn2branch-ecole/data/samples/setcover/500r_1000c_0.05d
STAMP=$(date +%Y%m%d_%H%M%S)
mkdir -p "$ART"
echo "checkpoint start $STAMP"
find "$SRC" -name 'sample_*.pkl' 2>/dev/null | wc -l | tee "$ART/partial_samples_${STAMP}_count.txt"
tar -C /kaggle/working/learn2branch-ecole/data/samples/setcover -czf "$ART/partial_samples_500r_1000c_0.05d_${STAMP}.tar.gz" 500r_1000c_0.05d
ls -lh "$ART"
echo "checkpoint done $STAMP"
